In [ ]:
SID4 = 9486
SEED = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0

PERSONAL_PARAMETERS = {
    'SID4': SID4,
    'SEED': SEED,
    'SLICE': SLICE,
    'HP_ID': HP_ID,
    'CLS_A': CLS_A,
    'CLS_B': CLS_B,
}

print('DATA 266 Homework 2 — Personal Parameters')
for name, value in PERSONAL_PARAMETERS.items():
    print(f'{name} = {value}')
print('\nSEED=9486 controls stochastic operations.')
print('SLICE=486 and HP_ID=0 are reported only and are not used to select Homework 2 experiments,')
print('because the assignment does not define an HW2 HP_ID mapping.')
print('CLS_A=6 and CLS_B=0 are reported only because the standing requirements require them.')


# DATA 266 — Homework 2: NLP and Deep Learning Optimization

This notebook will document three required parts: Word2Vec transfer learning on IMDB reviews, a LangChain RAG pipeline over ten movie documents, and controlled measurements of five optimization techniques. This scaffold intentionally contains no model downloads or experiments.

## 1. Personal Parameters and Reproducibility

The first executable cell reports all required personal parameters. `SEED=9486` controls stochastic operations. `SLICE=486` and `HP_ID=0` are reported only: Homework 2 does not define an HP_ID mapping, so neither value selects experiments. `CLS_A=6` and `CLS_B=0` are also reported only because they are standing requirements.

Run the following centralized setup cell before any future stochastic operation. For `PYTHONHASHSEED` to affect Python's hash randomization, set it before starting the kernel; the cell records and sets the intended value for reproducibility.

In [ ]:
# Run before future stochastic operations. Restart the kernel after setting PYTHONHASHSEED if needed.
import os
import random

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)

try:
    import numpy as np
    np.random.seed(SEED)
    print('NumPy seed set.')
except ImportError:
    print('Warning: NumPy is unavailable; NumPy seed was not set.')

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    print(f'PyTorch seed set; CUDA available: {torch.cuda.is_available()}')
except ImportError:
    print('Warning: PyTorch is unavailable; PyTorch/CUDA seeds were not set.')

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    print('TensorFlow seed set.')
except ImportError:
    print('Warning: TensorFlow is unavailable; TensorFlow seed was not set.')

print(f'PYTHONHASHSEED={os.environ["PYTHONHASHSEED"]}; Python random seed={SEED}')


## 2. Part 1 — Embedding-Based Transfer Learning

Embedding-based transfer learning starts from a general-purpose representation and continues training it on domain text. Here, pretrained Google News vectors provide the starting representation; IMDB reviews can then adapt shared word vectors toward movie-review usage.

Nearest neighbors are words with the highest cosine similarity to a target vector. This is different from own-vector movement: cosine similarity between a target's original and fine-tuned vector measures how much that word itself moved. Lower original-versus-fine-tuned similarity means greater movement.

This implementation is intentionally guarded. It reads only a local IMDB CSV and never initiates a download of `word2vec-google-news-300`. Results, figures, and artifacts are produced only from actual supplied vectors and executed training; no placeholder results are fabricated.

In [ ]:
# Part 1 configuration. Do not use SLICE or HP_ID to choose reviews or experiments.
from pathlib import Path

TARGET_WORDS = ['cast', 'score', 'plot', 'screen', 'review']
VECTOR_SIZE = 300
CONTEXT_WINDOW = 5
MIN_COUNT = 2
EPOCHS = 5
MAX_REVIEWS = None  # None uses all available reviews; set an explicit integer only when runtime requires it.
GENSIM_WORKERS = 1  # One worker is the most deterministic practical setting.
PRETRAINED_MODEL_PATH = None  # Optional local .bin/.bin.gz, KeyedVectors, or Word2Vec path.

TOKENIZATION_DESCRIPTION = (
    'Lowercase text; replace HTML <br>, <br/>, and <br /> tags with spaces; '
    'replace non-alphanumeric/non-apostrophe punctuation with spaces; split on whitespace.'
)
PART1_ARTIFACT_DIR = Path('artifacts/part1')
PART1_FIGURE_PATH = Path('figures/hw2/part1_embedding_shift_tsne.png')

print({
    'seed': SEED, 'target_words': TARGET_WORDS, 'vector_size': VECTOR_SIZE,
    'window': CONTEXT_WINDOW, 'min_count': MIN_COUNT, 'epochs': EPOCHS,
    'max_reviews': MAX_REVIEWS, 'workers': GENSIM_WORKERS,
    'tokenization': TOKENIZATION_DESCRIPTION,
})


### 2.1 Local IMDB Discovery and Cleaning

The discovery function checks the assignment's expected relative locations in order: `../IMDB Dataset.csv`, `../../IMDB Dataset.csv`, and `data/hw2/IMDB Dataset.csv`. The first CSV containing `review` and `sentiment` columns is used. No dataset is downloaded or copied. By default `MAX_REVIEWS=None` preserves all 50,000 reviews; any reduction must be explicitly configured and recorded.

In [ ]:
import re
import pandas as pd

EXPECTED_IMDB_PATHS = [
    Path('../IMDB Dataset.csv'),
    Path('../../IMDB Dataset.csv'),
    Path('data/hw2/IMDB Dataset.csv'),
]

def discover_imdb_csv(expected_paths=EXPECTED_IMDB_PATHS):
    """Return the first valid expected IMDB CSV and its DataFrame."""
    checked = []
    for candidate in expected_paths:
        candidate = candidate.expanduser()
        checked.append(str(candidate))
        if not candidate.is_file():
            continue
        frame = pd.read_csv(candidate)
        normalized_columns = {str(column).strip().lower(): column for column in frame.columns}
        required = {'review', 'sentiment'}
        if required.issubset(normalized_columns):
            frame = frame.rename(columns={
                normalized_columns['review']: 'review',
                normalized_columns['sentiment']: 'sentiment',
            })
            return candidate.resolve(), frame
        print(f'Skipping {candidate}: required review and sentiment columns were not both found.')
    raise FileNotFoundError(
        'No valid local IMDB CSV was found. Checked: ' + ', '.join(checked) + '. '
        'Provide the CSV at one of these expected locations; this notebook will not download it.'
    )

def tokenize_imdb_review(text):
    """Lowercase IMDB text, remove HTML line breaks, separate punctuation, and return word tokens."""
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'<br\s*/?\s*>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-z0-9']+", ' ', text)
    return [token for token in text.split() if token]

def load_tokenized_imdb(max_reviews=MAX_REVIEWS):
    dataset_path, imdb_frame = discover_imdb_csv()
    total_reviews = len(imdb_frame)
    if max_reviews is not None:
        if not isinstance(max_reviews, int) or max_reviews <= 0:
            raise ValueError('MAX_REVIEWS must be None or a positive integer.')
        imdb_frame = imdb_frame.iloc[:max_reviews].copy()
    used_reviews = len(imdb_frame)
    tokenized_reviews = [tokenize_imdb_review(review) for review in imdb_frame['review']]
    tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]
    dataset_info = {
        'dataset_path': str(dataset_path),
        'dataset_shape': list(discover_imdb_csv()[1].shape),
        'total_reviews': total_reviews,
        'used_reviews': used_reviews,
        'nonempty_tokenized_reviews': len(tokenized_reviews),
        'columns': list(imdb_frame.columns),
    }
    print('IMDB dataset path:', dataset_info['dataset_path'])
    print('IMDB dataset shape:', dataset_info['dataset_shape'])
    print('Reviews total / used / nonempty:', total_reviews, used_reviews, len(tokenized_reviews))
    return imdb_frame, tokenized_reviews, dataset_info


### 2.2 Guarded Pretrained Model Loading

Set `PRETRAINED_MODEL_PATH` to a local Google News binary, saved `KeyedVectors`, or saved `Word2Vec` model when one has been provided. If it is unset, the function also inspects standard Gensim cache locations but never calls `gensim.downloader.load`, so it cannot begin the approximately 1.6 GB download. If unavailable, it raises a clear provisioning message.

In [ ]:
def _import_gensim_components():
    try:
        from gensim.models import KeyedVectors, Word2Vec
        return KeyedVectors, Word2Vec
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            'Gensim could not be imported. This can indicate an incompatible Gensim/SciPy installation. '
            'Install compatible versions in the execution environment, then rerun; original error follows.'
        ) from exc

def candidate_google_news_paths(provided_path=None):
    """Return existing candidate paths without downloading or creating cache content."""
    candidates = []
    if provided_path:
        candidates.append(Path(provided_path).expanduser())
    gensim_roots = [Path.home() / 'gensim-data', Path.home() / '.cache' / 'gensim']
    filenames = [
        'word2vec-google-news-300.gz', 'word2vec-google-news-300.bin.gz',
        'word2vec-google-news-300.bin', 'word2vec-google-news-300.kv',
    ]
    for root in gensim_roots:
        for filename in filenames:
            candidates.append(root / 'word2vec-google-news-300' / filename)
    return [path for path in candidates if path.is_file()]

def load_pretrained_google_news(provided_path=PRETRAINED_MODEL_PATH):
    """Load a local Google News model only; never call a downloader."""
    KeyedVectors, Word2Vec = _import_gensim_components()
    available_paths = candidate_google_news_paths(provided_path)
    if not available_paths:
        raise FileNotFoundError(
            'word2vec-google-news-300 is not available locally. Set PRETRAINED_MODEL_PATH to a provided '
            'Google News binary, saved KeyedVectors, or Word2Vec model, or place it in the standard Gensim '
            'cache. This notebook deliberately does not download the approximately 1.6 GB model.'
        )
    model_path = available_paths[0]
    load_errors = []
    for loader_name, loader in [
        ('KeyedVectors.load', lambda: KeyedVectors.load(str(model_path), mmap='r')),
        ('Word2Vec.load', lambda: Word2Vec.load(str(model_path)).wv),
        ('load_word2vec_format', lambda: KeyedVectors.load_word2vec_format(
            str(model_path), binary=True, unicode_errors='ignore'
        )),
    ]:
        try:
            vectors = loader()
            if vectors.vector_size != VECTOR_SIZE:
                raise ValueError(f'Expected {VECTOR_SIZE}-dimensional vectors, found {vectors.vector_size}.')
            print(f'Loaded pretrained vectors using {loader_name}: {model_path}')
            return vectors, model_path.resolve(), loader_name
        except Exception as exc:
            load_errors.append(f'{loader_name}: {type(exc).__name__}: {exc}')
    raise RuntimeError(
        f'Could not load the provided local model at {model_path}. Attempts: ' + ' | '.join(load_errors)
    )


### 2.3 Fine-Tuning, Neighbor Analysis, Vector Movement, and Visualization

The original pretrained vectors are read-only inputs and are never overwritten. A fresh `Word2Vec` vocabulary is built from tokenized IMDB reviews, then every shared word receives a copied original vector before continued training. `seed=9486`, `workers=1`, and sorted vocabulary improve reproducibility; Gensim's training can still vary across versions and hardware.

The next cell is a function library rather than an automatic experiment. Invoke `run_part1()` only in an approved environment with a supplied model. It creates actual CSV/JSON/text artifacts and the t-SNE image only after successful execution.

In [ ]:
import json

def top_three_neighbors(vectors, words=TARGET_WORDS, stage='before'):
    rows = []
    for word in words:
        if word not in vectors.key_to_index:
            rows.extend({
                'word': word, 'rank': rank, f'neighbor_{stage}': None, f'similarity_{stage}': None
            } for rank in range(1, 4))
            continue
        neighbors = vectors.most_similar(word, topn=3)
        for rank, (neighbor, similarity) in enumerate(neighbors, start=1):
            rows.append({
                'word': word, 'rank': rank, f'neighbor_{stage}': neighbor,
                f'similarity_{stage}': float(similarity),
            })
    return pd.DataFrame(rows)

def build_finetuned_model(tokenized_reviews, pretrained_vectors):
    _, Word2Vec = _import_gensim_components()
    model = Word2Vec(
        vector_size=VECTOR_SIZE, window=CONTEXT_WINDOW, min_count=MIN_COUNT,
        workers=GENSIM_WORKERS, seed=SEED, sorted_vocab=1, epochs=EPOCHS,
    )
    model.build_vocab(tokenized_reviews)
    shared_words = [word for word in model.wv.index_to_key if word in pretrained_vectors.key_to_index]
    for word in shared_words:
        model.wv.vectors[model.wv.key_to_index[word]] = pretrained_vectors[word].copy()
    model.train(tokenized_reviews, total_examples=model.corpus_count, epochs=model.epochs)
    return model, len(shared_words)

def cosine_similarity(vector_a, vector_b):
    denominator = float(np.linalg.norm(vector_a) * np.linalg.norm(vector_b))
    if denominator == 0:
        return float('nan')
    return float(np.dot(vector_a, vector_b) / denominator)

def vector_shift_table(pretrained_vectors, finetuned_vectors, words=TARGET_WORDS):
    rows = []
    for word in words:
        if word in pretrained_vectors.key_to_index and word in finetuned_vectors.key_to_index:
            similarity = cosine_similarity(pretrained_vectors[word], finetuned_vectors[word])
            rows.append({
                'word': word,
                'cosine_similarity_original_vs_finetuned': similarity,
                'shift': 1.0 - similarity,
            })
    return pd.DataFrame(rows).sort_values('shift', ascending=False, ignore_index=True)

def save_embedding_shift_tsne(pretrained_vectors, finetuned_vectors, viz_word='score'):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    if viz_word not in pretrained_vectors.key_to_index or viz_word not in finetuned_vectors.key_to_index:
        raise KeyError(f'{viz_word!r} must be present in both models for the t-SNE visualization.')
    before_words = [viz_word] + [word for word, _ in pretrained_vectors.most_similar(viz_word, topn=5)]
    after_words = [viz_word] + [word for word, _ in finetuned_vectors.most_similar(viz_word, topn=5)]
    vectors = np.vstack([
        *[pretrained_vectors[word] for word in before_words],
        *[finetuned_vectors[word] for word in after_words],
    ])
    coordinates = TSNE(n_components=2, random_state=SEED, perplexity=5, init='pca').fit_transform(vectors)
    PART1_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(10, 8))
    n_before = len(before_words)
    axis.scatter(coordinates[:n_before, 0], coordinates[:n_before, 1], color='tab:blue', label='Before fine-tuning')
    axis.scatter(coordinates[n_before:, 0], coordinates[n_before:, 1], color='tab:orange', label='After fine-tuning')
    for index, word in enumerate(before_words):
        label = f'{word} (target, before)' if word == viz_word else f'{word} (before)'
        axis.annotate(label, coordinates[index], fontsize=8)
    for index, word in enumerate(after_words, start=n_before):
        label = f'{word} (target, after)' if word == viz_word else f'{word} (after)'
        axis.annotate(label, coordinates[index], fontsize=8)
    axis.set_title(f't-SNE embedding shift for {viz_word!r}: IMDB fine-tuning')
    axis.legend()
    figure.tight_layout()
    figure.savefig(PART1_FIGURE_PATH, dpi=200, bbox_inches='tight')
    plt.show()
    return PART1_FIGURE_PATH

def run_part1():
    """Run Part 1 only when a local pretrained model is available; save actual observed artifacts."""
    imdb_frame, tokenized_reviews, dataset_info = load_tokenized_imdb()
    pretrained_vectors, model_path, loader_name = load_pretrained_google_news()
    baseline_neighbors = top_three_neighbors(pretrained_vectors, stage='before')
    finetuned_model, shared_initialized_words = build_finetuned_model(tokenized_reviews, pretrained_vectors)
    finetuned_neighbors = top_three_neighbors(finetuned_model.wv, stage='after')
    neighbor_comparison = baseline_neighbors.merge(finetuned_neighbors, on=['word', 'rank'], how='outer')
    shifts = vector_shift_table(pretrained_vectors, finetuned_model.wv)
    if shifts.empty:
        raise RuntimeError('None of the required target words are shared by the original and fine-tuned vocabularies.')
    most_shifted, least_shifted = shifts.iloc[0], shifts.iloc[-1]
    figure_path = save_embedding_shift_tsne(pretrained_vectors, finetuned_model.wv, viz_word='score')
    PART1_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    configuration = {
        **dataset_info, 'seed': SEED, 'target_words': TARGET_WORDS, 'vector_size': VECTOR_SIZE,
        'context_window': CONTEXT_WINDOW, 'min_count': MIN_COUNT, 'epochs': EPOCHS,
        'max_reviews': MAX_REVIEWS, 'gensim_workers': GENSIM_WORKERS,
        'tokenization': TOKENIZATION_DESCRIPTION, 'pretrained_model_path': str(model_path),
        'pretrained_loader': loader_name, 'shared_vocabulary_words_initialized': shared_initialized_words,
        'tsne_random_state': SEED, 'tsne_figure_path': str(figure_path),
    }
    baseline_neighbors.to_csv(PART1_ARTIFACT_DIR / 'baseline_neighbors.csv', index=False)
    finetuned_neighbors.to_csv(PART1_ARTIFACT_DIR / 'finetuned_neighbors.csv', index=False)
    neighbor_comparison.to_csv(PART1_ARTIFACT_DIR / 'neighbor_comparison.csv', index=False)
    shifts.to_csv(PART1_ARTIFACT_DIR / 'vector_shift_metrics.csv', index=False)
    (PART1_ARTIFACT_DIR / 'part1_configuration.json').write_text(json.dumps(configuration, indent=2) + '\n')
    summary = (
        'Part 1 executed from actual local data and pretrained vectors.\n'
        f"Most shifted word: {most_shifted['word']}; similarity={most_shifted['cosine_similarity_original_vs_finetuned']:.6f}; shift={most_shifted['shift']:.6f}.\n"
        f"Least shifted word: {least_shifted['word']}; similarity={least_shifted['cosine_similarity_original_vs_finetuned']:.6f}; shift={least_shifted['shift']:.6f}.\n"
        'Lower original-versus-fine-tuned cosine similarity means greater own-vector movement.\n'
        'Limitation: corpus size, vocabulary overlap, Gensim/library versions, and stochastic training can affect results.\n'
    )
    (PART1_ARTIFACT_DIR / 'part1_run_summary.txt').write_text(summary)
    print(summary)
    return {
        'configuration': configuration, 'baseline_neighbors': baseline_neighbors,
        'finetuned_neighbors': finetuned_neighbors, 'neighbor_comparison': neighbor_comparison,
        'vector_shifts': shifts, 'most_shifted': most_shifted, 'least_shifted': least_shifted,
    }


### 2.4 Part 1 Interpretation and Limitations

After execution, compare the three nearest neighbors before and after training to describe changes in movie-review associations. Do not infer movement from neighbor similarity alone: use the target's original-versus-fine-tuned cosine similarity and `shift = 1 - similarity`. Results depend on the number and composition of reviews, shared-vocabulary coverage, tokenizer choices, model versions, and remaining stochasticity in Word2Vec training.

## 3. Part 2 — LangChain RAG

**Required deliverables:** load ten Wikipedia movie documents with a LangChain Document Loader; split them with chunk size 500 and overlap 50; create embeddings and a vector store; use explicit `PromptTemplate`, retriever, and LLM components; answer five questions with retrieved chunks; re-run two questions with alternate chunking; manually verify retrieval, report ranks and success rate, and analyze two RAG failures.

**Implementation placeholder:** use explicit, version-compatible LangChain components and ensure the described LLM exactly matches instantiated code. Save page metadata, retrieval evidence, chunks, answers, and failure analyses under `artifacts/part2/`. Do not fetch Wikipedia pages or download models during this scaffold phase.

## 4. Part 3 — Optimization Techniques

**Required deliverables:** controlled comparisons for CPU-versus-GPU tensor creation, weight initialization, activation checkpointing, gradient accumulation, and mixed-precision training. Record time, memory where available, and loss or another performance measure.

**Implementation placeholder:** pair each technique with a matched baseline, apply `SEED=9486` consistently, document device availability and measurement limitations, and avoid treating unavailable GPU measurements as results. Save raw output in `artifacts/part3/`, figures in `figures/`, and only compact intentional checkpoints in `checkpoints/`.

## 5. Reproducibility and Limitations

This notebook must use `SEED=9486` for stochastic operations, but exact numerical equivalence can still vary with library versions, hardware, GPU kernels, and live Wikipedia content. Record package versions, device details, input provenance, limits on review counts, and any unavailable resources in `RUN_LOG.txt`. Do not commit large model caches; store only required compact artifacts and cite their provenance.